# Week 3 exploration: ML, NLP and LLM foundations

Pipeline: **data -> preprocessing -> NLP/ML -> representation -> output**.

Run the cells from top to bottom. Cells marked *(downloads a model on first use)* need internet once.

In [ ]:
import sys
from pathlib import Path

# Make the project's src/ folder importable from inside notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))

from ai_foundations.preprocessing.text_cleaner import clean_text
print('Setup OK')

## 1. Preprocessing (Phase 02)
Raw text is messy. A cleaning function makes it consistent.

In [ ]:
examples = ['   I NEED HELP!!!   ', 'Hello,    WORLD!!', 'What are the admission requirements?']
for text in examples:
    print(repr(text), '->', repr(clean_text(text)))

## 2. Machine learning: training vs inference (Phase 01)
`fit` = **training** (learn from examples). `predict` = **inference** (use what was learned).

In [ ]:
from ai_foundations.ml.model import train_course_sentiment, predict_sentiment

vectorizer, model = train_course_sentiment()                      # training
print(predict_sentiment(vectorizer, model, 'I really love this course'))   # inference
print(vectorizer.get_feature_names_out())                         # the vocabulary it learned

## 3. Text classification (Phase 04)
Support messages -> category, using TF-IDF + Logistic Regression.

In [ ]:
import pandas as pd
from ai_foundations.config import SUPPORT_RAW_FILE

df = pd.read_csv(SUPPORT_RAW_FILE)
display(df.head(8))
print(df['label'].value_counts())

In [ ]:
from ai_foundations.nlp import classifier

classifier.prepare_dataset()                                  # raw -> cleaned
_, accuracy, report = classifier.train_classifier()          # evaluate on held-out data
print(f'Accuracy: {accuracy:.2f}')
print(report)

final = classifier.train_final_model()                       # retrain on everything
for message in ['I cannot reset my password', 'The server is not responding', 'When does the admission start?']:
    print(message, '->', classifier.predict_message(final, message))

## 4. Tokenization (Phase 03)
Text -> tokens -> token IDs.

In [ ]:
from ai_foundations.nlp import tokenizer

print(tokenizer.tokenize('Agentic AI systems can use tools'))
vocab = tokenizer.build_char_vocab('hello')
print(vocab, tokenizer.encode_chars('hello', vocab))
print(tokenizer.count_symbol_pairs('low lower lowest').most_common(3))   # first step of BPE

In [ ]:
from ai_foundations.config import MODELS_DIR

model_file = tokenizer.train_sentencepiece(tokenizer.SENTENCEPIECE_TRAINING_TEXT, MODELS_DIR)
result = tokenizer.sentencepiece_round_trip(model_file, 'Artificial intelligence is changing software engineering.')
print(result['pieces'])
print(result['token_ids'])
print(result['decoded'])

In [ ]:
# (downloads a model on first use)
bert = tokenizer.load_hf_tokenizer('bert-base-uncased')
info = tokenizer.inspect_text(bert, 'Large language models are powerful.')
for key, value in info.items():
    print(f'{key}: {value}')
print(tokenizer.special_tokens_info(bert))
print(tokenizer.compare_tokenizers('Artificial intelligence is transforming software engineering.', ['bert-base-uncased', 'gpt2']))

## 5. Embeddings and semantic search (Phases 05 and 06)
Similar *meaning* -> similar vectors, even with different words.

In [ ]:
# (downloads a model on first use, about 90 MB)
from ai_foundations.embeddings.embedder import Embedder

embedder = Embedder()
print('password pair:', round(embedder.similarity('How do I reset my password?', 'I forgot my password.'), 4))
print('weather pair: ', round(embedder.similarity('How do I reset my password?', 'The weather is very hot today.'), 4))

In [ ]:
# (uses the model loaded above)
documents = [
    'Students can register courses through the student portal.',
    'The examination office publishes examination schedules.',
    'The finance office handles tuition fee issues.',
    'The library provides digital research resources.',
]
embedder.index(documents)
for query in ['Where can I register my courses?', 'How do I pay my tuition?', 'When are the exam dates?']:
    print('\nQuery:', query)
    for document, score in embedder.search(query):
        print(f'  {score:.4f} -> {document}')

## 6. Reflection (answer in your own words)
1. What is the difference between training and inference?
2. Why can a model not use a word it never saw in training?
3. Why do two sentences with different words get a high similarity score?